# 00 · Data profiling, business context & preprocessing

> **Public copy (NDA).** Outputs are cleared and private values are removed from the text. Run it locally with the NDA data in `data/`; never push it with outputs. Figures are not saved from this copy (`save` is a no-op): the public figures come from `public/reports/make_public_figures.py`.


**Schneider Electric × X-HEC Data Challenge 2026.** We forecast raw-material (RM) prices **3, 6 and 9 months ahead** for two medium-voltage Ring Main Units (RMUs) and turn the forecasts into a sourcing and pricing recommendation.

| Product | Main materials |
|---|---|---|
| **RM AirSeT IIQ 24 kV** (SF6-free) | carbon steel, stainless steel, plastics, aluminium |
| **RM6 NE-IQI** (SF6) | carbon steel, stainless steel, copper, galvanised steel, PET, aluminium |

**Why procurement cares.** RMUs are sold as *engineer-to-order* (ETO) contracts with prices locked for 3, 6 or 9 months. Between quote and purchase, raw-material prices can move against Schneider, and the margin absorbs the difference. A good forecast plus a calibrated range is what lets you:
- price a quote correctly;
- decide when to forward-buy or hedge;
- design the index clauses in supplier contracts.

**What this notebook does.** It goes through every file we received. Each file gets the same blocks:

| Block | Content |
|---|---|
| **(a) Business** | What the file is, who produces it, and which product or material it feeds |
| **(b) Columns** | Every column: dtype, non-null count, missing %, cardinality, range, examples |
| **(c) Missing values** | Licensed or empty columns, publication-lag ragged edge, internal holes |
| **(d) Duplicates** | Duplicate rows, duplicate keys, identical columns or files, near-duplicate series |
| **(e) Interesting** | Outliers, shocks, conflicts, quick plot |
| **(f) Decision** | What the preprocessing keeps, drops or fixes |

The final section builds the **model-ready parquet files** in `data/processed/` with [`src/preprocessing.py`](../src/preprocessing.py).

**Rules from the guidelines** (`data/raw/data/README.md`):
- missing values are blank, never 0, and never interpolated by the provider: *"imputation is part of the challenge"*;
- L01–L08 are licensed series and are empty on purpose;
- the last 2–3 Eurostat/BLS points can be revised;
- the last 3 rows of the BOM are totals, not materials;
- indices have different base years, so compare growth rates, not levels;
- S01–S05 are in USD; convert them with S06.

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib as mpl
from IPython.display import display, Markdown
import src.data_loading as dl
import src.features as fe
import src.preprocessing as pp

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 60); pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
INK, INK2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
mpl.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "axes.titlecolor": INK, "axes.titlesize": 11,
    "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRID, "grid.linewidth": 0.6,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.prop_cycle": mpl.cycler(color=PAL), "lines.linewidth": 1.6, "legend.frameon": False,
    "legend.fontsize": 8, "font.family": "sans-serif", "figure.dpi": 110,
})

def plot_missing(wide, title, max_cols=60):
    # observed = dark cell, missing = light cell; one row per series
    w = wide.iloc[:, :max_cols]
    fig, ax = plt.subplots(figsize=(11, max(2.2, 0.18 * w.shape[1])))
    ax.imshow(w.notna().T.values, aspect="auto", interpolation="nearest",
              cmap=mpl.colors.ListedColormap(["#f0efe9", PAL[0]]))
    ax.set_yticks(range(w.shape[1])); ax.set_yticklabels(w.columns, fontsize=6.5)
    step = max(1, len(w) // 12)
    ax.set_xticks(range(0, len(w), step)); ax.set_xticklabels(w.index[::step].strftime("%Y-%m"), fontsize=7)
    ax.grid(False); ax.set_title(title); plt.show()

def rebased(df, base=None):
    base = base or df.apply(lambda s: s.first_valid_index()).max()
    return df.div(df.loc[base]).mul(100)

dl.extract_archives()
print("Data folder:", dl.DATA)

## 1 · Inventory of every file

The organisers sent two zips (`data.zip`, `Supplier_balancesheet.zip`, extracted to `data/raw/`), eight loose workbooks and two PDFs. The table below has one row per (file, sheet).

In [ ]:
inv = dl.file_inventory()
inv

In [ ]:
# Byte-identical files anywhere in data/ (md5)
dups = pp.duplicate_files()
print("Byte-identical file groups:", "none" if dups.empty else "")
display(dups)
# The root RM_Prices_XHEC_Challenge.xlsx is NOT the same file as the one inside data.zip
a, b = dl.DATA / "RM_Prices_XHEC_Challenge.xlsx", dl.CORE / "RM_Prices_XHEC_Challenge.xlsx"
print("root vs data.zip RM_Prices_XHEC_Challenge.xlsx identical:", pp.file_md5(a) == pp.file_md5(b),
      "| sheets root:", pd.ExcelFile(a).sheet_names, "| sheets zip:", pd.ExcelFile(b).sheet_names)

**What to keep in mind.**
- No two files are byte-identical. There is a lot of *logical* redundancy, though, and the sections below check each case:
  - the Excel masters inside `data.zip` repeat the CSVs;
  - each `long` CSV repeats its `wide` CSV;
  - `E23` repeats `S06`;
  - the ▲ sheets of the steel workbook are differences of its price sheets.
- The root `RM_Prices_XHEC_Challenge.xlsx` has **only a BOM sheet**, and that BOM conflicts with the one in `data.zip` (§4).

## 2 · `rm_prices_monthly_wide.csv` / `_long.csv` + `source_catalog.csv` + `data_dictionary.csv`

### (a) Business
This is the **core raw-material panel** prepared by the organisers. Every series is free and public (IMF and BLS via FRED, ECB, Eurostat). There are three kinds of series:

| Prefix | Meaning | Series |
|---|---|---|
| **S** | free source, filled | **LME metals in USD** (exchange prices, the world benchmark): S01 copper, S02 aluminium, S03 nickel (cost driver of stainless 304L, ≈8–10 % Ni), S05 zinc (galvanising coating) · S04 **iron ore** (carbon-steel driver) · S06 **EUR/USD** · **Eurostat producer-price indices** (S07 copper, S08 aluminium, S09 iron & steel, S10 German steel as an HDG proxy, S11 plastics) · **US BLS PPIs** (S12 steel, S13 plastics) |
| **D** | derived in Excel | D01 = S01/S06 copper in EUR/t; D02 = S02/S06 aluminium in EUR/t. These are the EU buyer's view. |
| **L** | licensed, **empty on purpose** | CRU/MEPS/Fastmarkets/ICIS prices for CR coil, 304L, HDG, PBT, PET, ETP and the Rotterdam Al/Cu premiums. These are the prices Schneider actually pays. **The steel workbook (§6) fills most of this gap.** |

**Prices vs indices.** PPIs (Index 2021 = 100) are what EU producers charge on average. They are smooth and sticky, and supply contracts are often indexed on them. Exchange prices (LME) move daily and are close to a random walk.

In [ ]:
rm_raw = pd.read_csv(dl.CORE / "rm_prices_monthly_wide.csv")
rm_long = pd.read_csv(dl.CORE / "rm_prices_monthly_long.csv")
rm_cat = pd.read_csv(dl.CORE / "source_catalog.csv")
ddict = pd.read_csv(dl.CORE / "data_dictionary.csv")
print("wide:", rm_raw.shape, "| long:", rm_long.shape, "| catalog:", rm_cat.shape, "| dictionary:", ddict.shape)
rm_cat

### (b) All columns: wide file (profile + catalog description)

In [ ]:
prof = pp.profile(rm_raw).join(rm_cat.set_index("series_id")[["material", "region", "unit", "proxy_type", "access"]])
prof

In [ ]:
print("Long file columns:"); display(pp.profile(rm_long))
print("Data dictionary (field rules given by the organisers):"); display(ddict)

### (c) Missing values

In [ ]:
rm = dl.load_rm_wide()
cov = pp.series_coverage(rm)
display(cov)
plot_missing(rm, "rm_prices_monthly_wide · observed (blue) vs missing; L01–L08 licensed = fully empty")
print("Series 100% empty:", cov.index[cov.n_obs == 0].tolist())
print("Series with internal holes:", cov.index[cov.internal_nan > 0].tolist() or "none")

- **L01–L08 are 100 % empty**, which the guidelines say is on purpose (licensed).
- **There are no internal holes.** The only gap is the **ragged right edge**: every S/D series ends in **Sep-2026** except FX (S06), which reaches Oct-2026. That is the publication lag.
- So the "imputation" problem is a **nowcasting** problem: filling the last month.

### (d) Duplicates

In [ ]:
display(pp.duplicate_report(rm_raw, keys=["month"]).to_frame("wide"))
display(pp.duplicate_report(rm_long, keys=["month", "series_id"]).drop("identical_column_pairs").to_frame("long"))

# long <-> wide round trip: is the long file exactly the stacked wide file?
lw = rm_long.assign(month=pd.to_datetime(rm_long.month)).pivot(index="month", columns="series_id", values="value")
w = rm[lw.columns].loc[lw.index]
print(f"long vs wide ({lw.shape[1]} series): same NaN pattern = {(lw.isna() == w.isna()).all().all()},",
      f"max relative difference = {(lw / w - 1).abs().max().max():.1e} (CSV rounding) | long has D01/D02? {({'D01', 'D02'} <= set(rm_long.series_id))}")

# D01/D02 are pure identities
for d, s in [("D01", "S01"), ("D02", "S02")]:
    print(f"{d} == {s}/S06 : max rel. error {((rm[d] - rm[s] / rm.S06) / rm[d]).abs().max():.2e}")
print("\nNear-duplicate series (corr of monthly % changes ≥ 0.98):")
display(pp.near_duplicate_columns(rm))

- **No duplicate rows or months.** The long file is the stacked wide file (S series only, same values up to rounding of ≈ 5·10⁻⁵), so it is redundant.
- **D01/D02 are identities** (S01/S06, S02/S06, exact up to the 4-decimal rounding of S06). We keep them because the EUR price is what an EU buyer pays. A model must not use the USD price and D01 as two independent signals.

### (e) Interesting things

In [ ]:
key = ["D01", "D02", "S03", "S04", "S05", "S07", "S08", "S09", "S10", "S11", "S12", "S13"]
idx = rebased(rm[key], "2016-11-01")
fig, ax = plt.subplots(figsize=(11, 4))
for i, c in enumerate(["D01", "D02", "S03", "S04", "S09", "S11"]):
    ax.plot(idx.index, idx[c], color=PAL[i], label=f"{c} · {rm_cat.set_index('series_id').material.get(c, c)}")
ax.axhline(100, color=AXIS, lw=0.8); ax.legend(ncol=3); ax.set_title("Raw materials rebased Nov-2016 = 100")
plt.show()

lr = np.log(rm[key]).diff()
stats = pd.DataFrame({
    "last": rm[key].apply(lambda s: s.dropna().iloc[-1]),
    "YoY_%": 100 * (rm[key].ffill().iloc[-1] / rm[key].ffill().iloc[-13] - 1),
    "peak_month": rm[key].idxmax().dt.strftime("%Y-%m"),
    "last_vs_peak_%": 100 * (rm[key].ffill().iloc[-1] / rm[key].max() - 1),
    "ann_vol_%": lr.std() * np.sqrt(12) * 100,
    "AC1_dlog": lr.apply(lambda s: s.autocorr(1)),
    "max_abs_monthly_move_%": 100 * lr.abs().max(),
    "month_of_max_move": lr.abs().idxmax().dt.strftime("%Y-%m"),
})
stats.round(2)

**Takeaways for the business.**
- **Copper (D01) is at an all-time high** (peak month = the last month). It is the main risk for RM6, which contains a significant amount of copper.
- Nickel peaked in the **Mar-2022 LME squeeze**, the largest single monthly move. Steel PPIs peaked in mid-2022 with the energy shock.
- **Two regimes:** PPIs have strong momentum (AC1 ≈ 0.7–0.8, low volatility). LME metals are close to a random walk (AC1 ≈ 0.1–0.3, high volatility). Plan the modelling around that.

### (f) Decision
- Keep S01–S13 and D01–D02 at month-start frequency.
- **Drop L01–L08** (empty).
- Drop `year`/`month_num`, since they can be rebuilt from the index.
- Do not use the long file (redundant).
- Ragged edge → nowcast:
  1. S01/S02 from the LME workbook (§7), then D01/D02 by identity;
  2. the others by forward-fill of ≤ 3 months, flagged.

## 3 · `energy_prices_monthly_wide.csv` / `_long.csv` + `energy_source_catalog.csv`

### (a) Business
Energy is the largest **variable cost** behind the materials, and it moves *before* them. That makes it a natural **leading indicator**:
- **aluminium smelting** is power-hungry: EU smelters curtailed output in 2022;
- **steel** uses gas for reheating and power for electric-arc furnaces;
- **plastics** are made from oil and gas (paraxylene, naphtha).

The file covers four regions:
- **Europe**: power in DE, FR and HU;
- the **US** and **Japan/Asia**, the competing cost regions;
- **Russia**, through its share of EU crude imports, which captures the 2022 cut-off.

`E` series are downloaded. `ED` series are harmonised to EUR/MWh or EUR/bbl, so gas and power can be compared across regions.

In [ ]:
en_raw = pd.read_csv(dl.CORE / "energy_prices_monthly_wide.csv")
en_long = pd.read_csv(dl.CORE / "energy_prices_monthly_long.csv")
en_cat = pd.read_csv(dl.CORE / "energy_source_catalog.csv")
print("wide:", en_raw.shape, "| long:", en_long.shape, "| catalog:", en_cat.shape)
en_cat[["series_id", "energy", "region", "series_name", "provider", "unit", "relevance", "first_month", "last_month"]]

### (b) All columns

In [ ]:
pp.profile(en_raw).join(en_cat.set_index("series_id")[["energy", "region", "unit"]])

### (c) Missing values

In [ ]:
en = dl.load_energy_wide()
display(pp.series_coverage(en)[["first", "last", "n_obs", "internal_nan", "trailing_nan"]].T)
plot_missing(en, "energy_prices_monthly_wide · ragged right edge only")

There are no internal holes, and the ragged edge has three steps:
- **Oct-2026:** power (Ember, JEPX), Bank of Japan series and FX are available;
- **Sep-2026:** IMF and Eurostat PPIs stop here;
- **Jul-2026:** Eurostat crude imports stop here (E20, E21, ED06), about 3–4 months behind.

### (d) Duplicates

In [ ]:
display(pp.duplicate_report(en_raw, keys=["month"]).to_frame("wide"))
print("E23 == S06 (USD per EUR):", np.allclose(en.E23, rm.S06.reindex(en.index), equal_nan=True))
checks = {"ED01 = E01/E23": en.E01 / en.E23, "ED02 = E05·3.412/E23": en.E05 * 3.412 / en.E23,
          "ED03 = E07·3.412/E23": en.E07 * 3.412 / en.E23, "ED04 = E06·3.412/E23": en.E06 * 3.412 / en.E23,
          "ED05 = E12·1000/E24": en.E12 * 1000 / en.E24, "ED06 = E20/E21": en.E20 / en.E21}
print(pd.Series({k: f"max rel. err {((en[k[:4]] - v) / en[k[:4]]).abs().max():.1e}" for k, v in checks.items()}).to_string())
lw = en_long.assign(month=pd.to_datetime(en_long.month)).pivot(index="month", columns="series_id", values="value")
w = en[lw.columns].loc[lw.index]
print(f"long vs wide: same NaN pattern = {(lw.isna() == w.isna()).all().all()}, max relative difference by series (top 3):",
      (lw / w - 1).abs().max().sort_values().tail(3).map("{:.1e}".format).to_dict())
print("\nNear-duplicate series (corr of monthly % changes ≥ 0.98):")
display(pp.near_duplicate_columns(en))

- **E23 is an exact copy of S06**, so we drop E23 from the panel.
- The **ED** columns are transformations of the E columns: exact up to rounding (≈ 10⁻⁴; ED06 ≈ 1 % because a small share is rounded to 4 decimals). They are useful because their units are comparable, but they carry no extra information. The near-duplicate table shows exactly these E ↔ ED pairs.
- The long file is the stacked wide file (same values up to rounding), so it is redundant.
- Brent, WTI and Dubai crude are highly collinear (correlation of monthly changes 0.83–0.93). Pick **one oil feature** per model.

### (e) Interesting things

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
for i, (c, lab) in enumerate([("ED02", "TTF gas EU"), ("E09", "Power DE"), ("E10", "Power FR"),
                              ("E11", "Power HU"), ("ED04", "Henry Hub US")]):
    ax.plot(en.index, en[c], color=PAL[i], label=lab)
ax.set_ylabel("EUR/MWh"); ax.legend(ncol=5); ax.set_title("2022 energy shock: EU gas and power vs US gas")
plt.show()
print("Russia share of EU crude imports (ED06), yearly mean:")
print((en.ED06.groupby(en.index.year).mean() * 100).round(1).to_string())
print("\nLatest month vs previous month, power (EUR/MWh):")
print(en[["E09", "E10", "E11"]].dropna().tail(3).round(1).to_string())

- In 2022 European gas and power went up about 10×, while US gas barely moved. This explains the EU steel and aluminium PPI peak, and it is the **structural cost handicap of EU plants**.
- **Hungarian power is currently rising fast.** For EU production this is a conversion-cost risk, not just a leading indicator.
- **Russia's share of EU crude imports** fell from about 30 % to about 1 %, a clean regime-change marker.

### (f) Decision
- Keep E01–E24 and ED01–ED06, **minus E23** (a duplicate of S06).
- Nowcast the ragged edge:
  - ED identities where their components exist;
  - otherwise forward-fill of ≤ 3 months, flagged.
- The 3-month gap of E20/E21/ED06 is filled the same way. Treat these as slow structural features, not as short-term signals.

## 4 · `bom_materials.csv` (Schneider internal) vs root `RM_Prices_XHEC_Challenge.xlsx`

### (a) Business
The **bill of materials** turns material prices into a **product material cost**: cost(t) = Σ mass_kg × price_EUR/kg(t). It is the bridge between a price forecast and a P&L impact per unit. This is the only Schneider-internal file in `data.zip`. It was read off pie charts, so some slices are unlabelled.

In [ ]:
bom_raw = pd.read_csv(dl.CORE / "bom_materials.csv")
bom_raw

In [ ]:
display(pp.profile(bom_raw))
mat = bom_raw[bom_raw.plant_code.notna()]
print("Rows that are totals, not materials (to drop):", bom_raw.loc[bom_raw.plant_code.isna(), "product_ref"].tolist())
print("Duplicate (product, material label):", mat.duplicated(["product_ref", "material_label_source"]).sum())
impl = (mat.mass_kg / mat.share_pct).groupby(mat.product_ref).agg(["min", "max"]).round(1)
print("Implied total mass mass_kg/share_pct (should be constant per product):"); display(impl)
print("Labelled share of mass:", mat.groupby("product_ref").share_pct.sum().round(3).to_dict())

In [ ]:
# The other BOM: the root workbook has ONLY a BOM sheet, with different masses
root = pd.read_excel(dl.DATA / "RM_Prices_XHEC_Challenge.xlsx", sheet_name="BOM_Materials")
cmp = (mat[["product_ref", "material_label_source", "share_pct", "mass_kg"]]
       .merge(root[["product_ref", "material_label_source", "share_pct", "mass_kg"]],
              on=["product_ref", "material_label_source"], how="outer", suffixes=("_zip", "_root")))
cmp["mass_diff_kg"] = cmp.mass_kg_root - cmp.mass_kg_zip
cmp[cmp.mass_diff_kg.abs() > 0.05]

**Interesting and important.**
- **Two conflicting BOMs.** For AirSeT, the root workbook gives more carbon steel and less stainless than `data.zip`. We use the **`data.zip` version**: it is documented in the README and is the newer one. This needs confirming with the organisers.
- **RM6: only about 41 % of the mass is labelled.** The largest slice is "carbon steel, to confirm" and has **no mass**. Without it, RM6 material cost is understated by roughly the whole carbon-steel bill.
- **AirSeT: a sizeable "Other" share** cannot be priced.
- RM6 contains copper; AirSeT has none in this BOM version (the latest BOM v1 adds some). So the products have very different risk profiles.

### (f) Decision
`bom.parquet` holds:
- the materials only (the 3 total rows are dropped);
- `implied_total_mass_kg`;
- `model_series`, the price series used for each material;
- `mass_kg_model`, where RM6 carbon steel = an assumed share × implied total (from `features.bom_masses()`), with `is_assumption = True`;
- `mass_kg_root_xlsx`, kept for traceability.

## 5 · Excel masters inside `data.zip` (`RM_Prices_XHEC_Challenge.xlsx`, `Energy_Prices_XHEC_Challenge.xlsx`)

### (a) Business
These are the workbooks the organisers filled from the public APIs. The CSVs were exported from them. We only need to check that nothing in them is missing from the CSVs.

In [ ]:
for wb, csv, extra in [("RM_Prices_XHEC_Challenge.xlsx", "rm_prices_monthly_wide.csv", ["year", "month_num"]),
                       ("Energy_Prices_XHEC_Challenge.xlsx", "energy_prices_monthly_wide.csv", [])]:
    xl = pd.ExcelFile(dl.CORE / wb)
    x = pd.read_excel(xl, "Monthly_Wide")
    c = pd.read_csv(dl.CORE / csv)
    x["month"] = dl._month(x["month"]); c["month"] = dl._month(c["month"])
    num = [k for k in c.columns if k != "month" and k not in extra]
    m = x.set_index("month")[num].apply(pd.to_numeric, errors="coerce")
    cc = c.set_index("month")[num]
    print(f"{wb}: sheets={xl.sheet_names}\n   Monthly_Wide {x.shape} vs CSV {c.shape}: same NaN pattern = {(m.isna() == cc.isna()).all().all()},",
          f"max relative difference = {(m / cc - 1).abs().max().max():.1e}")
print("\nREADME sheet of the RM workbook (first lines):")
print(pd.read_excel(dl.CORE / "RM_Prices_XHEC_Challenge.xlsx", "README", header=None).iloc[:8, 0].to_string())

**(d) Duplicates / (f) Decision.** The `Monthly_Wide` sheets match the CSVs: same shape and NaN pattern, and the same values up to CSV rounding (ED06 is the 1 % outlier, explained in §3). D01/D02 hold Excel formulas, and the CSV stores their computed values. The README, Source_Catalog and Data_Dictionary sheets repeat the CSV and Markdown documentation. **We ignore these workbooks** and read the CSVs.

## 6 · `Copia de Europe Steel INDEXES.xlsx`: MEPS & CRU steel prices (Schneider procurement working file)

### (a) Business
This is the **most valuable file**. It contains the **licensed market prices** that the core set left empty (L01–L03), as Schneider's steel buyers use them.
- **CRU** and **MEPS** are price-reporting agencies. Each month they publish transaction prices for steel products by country.
- **Flat products** are what RMU enclosures are made of:
  - **HR** (hot-rolled coil): the upstream product;
  - **CR** (cold-rolled): enclosures and panels;
  - **HDG** (hot-dip galvanised): corrosion-protected parts;
  - **EG/EZ** (electro-galvanised).
- **Stainless 304/316/430** is priced as a **base price plus a monthly alloy surcharge**. Mills compute the surcharge from the previous period's LME nickel, chromium and molybdenum prices.
- **Reference to Index Mapping** links each of several hundred Schneider part references to the index that its supplier contract is indexed on (country × grade), with a **lead time**. It is the purchasing team's own language.

The workbook has 13 sheets. We go through them one at a time.

In [ ]:
xl = pd.ExcelFile(dl.STEEL_XLSX)
pd.DataFrame([(s, *pd.read_excel(xl, s, header=None).shape) for s in xl.sheet_names], columns=["sheet", "rows", "cols"])

### 6.1 Sheet `MEPS`: flat-coil "low" prices by country
**(b) Layout.** Multi-row header: row 0 holds the country, row 1 the unit text, row 4 the product. `x` columns are visual separators. Dates are month-start. `dl.load_meps()` parses it into long format.

In [ ]:
raw = pd.read_excel(dl.STEEL_XLSX, "MEPS", header=None)
print("raw header block:"); display(raw.iloc[:6, :12])
meps = dl.load_meps()
display(pp.profile(meps))
cov = meps.groupby(["country", "product", "unit"]).month.agg(first="min", last="max", n="count").reset_index()
cov["expected"] = [(l.year - f.year) * 12 + l.month - f.month + 1 for f, l in zip(cov["first"], cov["last"])]
cov["internal_holes"] = cov.expected - cov.n
cov

In [ ]:
print("Duplicate (country, product, unit, month):", meps.duplicated(["country", "product", "unit", "month"]).sum())
be = meps.query("country == 'Belgium' and product == 'HDG'").set_index("month").value
full = pd.date_range(be.index.min(), be.index.max(), freq="MS")
print("Belgium missing month(s):", full.difference(be.index).strftime("%Y-%m").tolist())
print("Empty placeholder rows after the last data (to 2026-12):", raw.iloc[5:, 0].notna().sum() - meps.month.nunique())

- **(c)** Belgium (all 4 products) has **one internal missing month**. Belgium matters because **HDG Benelux is the most-used index in the mapping**. The filled panel interpolates that month linearly and flags it.
- **Nordic** (30 months) and **Sweden** (18 months, quoted in both EUR and SEK, so two blocks with the same country name) are short fragments. We drop them.
- UK is quoted in **GBP**. Keep it, but never mix it with EUR series without converting.

### 6.2 Sheet `CRU Flat`: CRU weekly and monthly-average flat prices
**(b) Layout.** For DE, FR and IT there are 5 weekly columns plus an `AVERAGE` column per product. For ES and UK there is a single `Spot price` column. Italy also has **Import CIF/DDP** HR prices, which start only in 2026.

In [ ]:
raw = pd.read_excel(dl.STEEL_XLSX, "CRU Flat", header=None)
hdr = raw.iloc[1:6].T.iloc[1:]; hdr.columns = ["kind", "product", "country", "market", "unit"]
display(hdr.groupby(["country", "market", "kind"]).size().unstack(fill_value=0))
cru = dl.load_cru_flat()
display(pp.profile(cru))
cru.groupby(["country", "product", "market", "unit"]).month.agg(first="min", last="max", n="count")

In [ ]:
# Is AVERAGE the mean of the weekly quotes?  (DE CR coil = columns 7..12)
body = raw.iloc[6:].copy(); body[0] = dl._month(body[0]); body = body.dropna(subset=[0]).set_index(0)
wk = body.iloc[:, 6:11].apply(pd.to_numeric, errors="coerce"); avg = pd.to_numeric(body.iloc[:, 11], errors="coerce")
d = (wk.mean(axis=1) - avg).dropna()
print(f"DE CR coil: AVERAGE - mean(weeks): median {d.median():.2f}, max |diff| {d.abs().max():.2f} EUR/t over {len(d)} months")
print("Duplicate (country, product, market, month):", cru.duplicated(["country", "product", "market", "month"]).sum())
print("Months after Oct-2026 (beyond the challenge window):")
display(cru[cru.month > "2026-10-01"].groupby(["country", "product"]).value.last().unstack())

In [ ]:
# MEPS vs CRU for the same product/country: near-duplicates?
pairs = []
for ctry, cc in [("Germany", "DE"), ("France", "FR"), ("Italy", "IT"), ("Spain", "ES")]:
    for prod in ["HR", "CR", "HDG"]:
        a = cru.query("country == @ctry and product == @prod and market == 'Domestic'").groupby("month").value.mean()
        b = meps.query("country == @ctry and product == @prod").groupby("month").value.mean()
        j = pd.concat([a, b], axis=1, keys=["cru", "meps"]).dropna()
        pairs.append({"pair": f"{prod} {cc}", "n": len(j), "corr_levels": j.corr().iat[0, 1],
                      "corr_dlog": np.log(j).diff().corr().iat[0, 1], "mean_gap_%": 100 * (j.meps / j.cru - 1).mean()})
pd.DataFrame(pairs).set_index("pair").round(3)

- **(d)** CRU and MEPS **agree almost perfectly in levels** (correlation ≈ 1.00) and strongly in monthly changes. They are **near-duplicates**. Use one as the target; the other can serve as a robustness check or a feature.
- `AVERAGE` is the mean of the weekly quotes, so the weekly columns add no information at monthly frequency.
- **(e)** FR, ES and UK already have **Nov-2026** quotes, one month ahead of the challenge window. They are useful for nowcasting and are stored in `beyond_panel_end.parquet`.
- EG coil (stopped 2017/2020) and Italy imports (10 months) are dropped.

### 6.3 Sheet `CRU Stainless`: base price + alloy surcharge
For 304, 430 and 316 CR-2mm, by country, there are two columns: `BASE` and `Alloy`. The **transaction price = base + alloy**. Dates are mid-month (the 15th) and are normalised to month-start.

In [ ]:
ss = dl.load_cru_stainless()
display(pp.profile(ss))
display(ss.groupby(["country", "product", "unit"]).month.agg(first="min", last="max", n="count"))
print("Duplicate (country, product, month):", ss.duplicated(["country", "product", "month"]).sum())
de = ss.query("country == 'Germany' and product == 'SS304'").set_index("month")
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.stackplot(de.index, de.base, de.alloy, colors=[PAL[0], PAL[3]], labels=["base", "alloy surcharge"], alpha=0.85)
ax.legend(loc="upper left"); ax.set_ylabel("EUR/t"); ax.set_title("CRU stainless 304 Germany = base + alloy surcharge")
plt.show()
ni = rm.S03 / rm.S06
j = pd.concat({"alloy": de.alloy, "ni_eur": ni}, axis=1).dropna()
print("corr Δlog(alloy surcharge) vs Δlog(nickel EUR) at lag 0/1/2:",
      [round(np.log(j.alloy).diff().corr(np.log(j.ni_eur).diff().shift(k)), 2) for k in (0, 1, 2)])
print("alloy share of 304 DE price, mean / min / max: ",
      (de.alloy / de.value).agg(["mean", "min", "max"]).round(2).tolist())

- **(e)** The **alloy surcharge follows nickel with a one-month lag**, because the surcharge formula uses the previous month's LME price. That makes it close to **deterministic one month ahead**. Forecast the base price and the surcharge separately.
- **(c)** DE and FR run to **Aug-2026**. ES, IT, UK and all 316/430 grades stop at **Feb-2026**. Those are flagged as *lagging* and are not forward-filled for 8 months.

### 6.4 Sheet `CRU Long`: long products (beams, merchant bar, rebar, wire rod)
These are construction steel products. They are **not in the RMU BOM**, but they are a broad steel-cycle signal. History goes back to 1997, and the dates are mid-month.

In [ ]:
lg = pp.load_cru_long()
display(pp.profile(lg))
c = lg.groupby(["country", "product"]).month.agg(first="min", last="max", n="count").reset_index()
c["status"] = np.where(c["last"] < "2020-01-01", "discontinued", "live")
display(c.pivot(index="product", columns="country", values="last").astype(str).replace("NaT", ""))
print("Duplicate (country, product, month):", lg.duplicated(["country", "product", "month"]).sum())

Several French, Spanish and UK series **stop in Mar-2017**. The panel keeps them, since they are history-only drivers, and the catalog marks them `status = discontinued`. They are never imputed.

### 6.5 The ▲ sheets (`F▲ M -(M-1)`, `F▲ Q -(Q-1)`, `L▲ M -(M-1)`, `SS▲ M -(M-1)`)
These hold **month-on-month or quarter-on-quarter price changes in EUR/t**. They are the numbers fed into **price-adjustment clauses**: the supplier price moves by Δindex. Are they just differences of the price sheets?

In [ ]:
dm = pp.load_delta_sheet("F▲ M -(M-1)")
dm["month"] = dl._month(dm.period)
display(pp.profile(dm))
rows = []
for prod, ctry in [("HR", "Germany"), ("CR", "Germany"), ("HDG", "Germany"), ("CR", "France")]:
    p = meps.query("country == @ctry and product == @prod").set_index("month").value.diff()
    d = dm.query("product == @prod and country == @ctry and source == 'MEPS'").set_index("month").delta
    j = pd.concat([d, p], axis=1, keys=["sheet_delta", "recomputed"]).dropna()
    rows.append({"series": f"MEPS {prod} {ctry}", "n": len(j), "share_equal": (j.sheet_delta.round(2) == j.recomputed.round(2)).mean(),
                 "max_abs_diff": (j.sheet_delta - j.recomputed).abs().max()})
display(pd.DataFrame(rows))
print("Rows dated after the last observed price (placeholder 0s):", (dm.month > "2026-10-01").sum())
for s in ["F▲ Q -(Q-1)", "L▲ M -(M-1)", "SS▲ M -(M-1)"]:
    d = pp.load_delta_sheet(s); print(f"{s}: {d.shape}, products={d['product'].unique().tolist()[:6]}")

- **(d)** The ▲ sheets are **exact differences** of the MEPS/CRU price sheets: 100 % of the tested months match. **They are redundant for modelling.**
- **(e)** Months after the last price are filled with **0**, not blank. A naive reader would see "no change". **Never read future 0s as data.**
- **(f)** They are not exported. We model price levels or log-changes and can always recompute the deltas.

### 6.6 Sheet `Reference to Index Mapping`: Schneider part references → index
**(a)** For each part (sheet, coil, blank) the sheet gives the plant, supplier allocation, grade, dimensions, unit weight and the **CRU/MEPS index its price is linked to**, plus the clause **lead time**. It also holds historical supplier offer prices and their monthly updates (not shown).

In [ ]:
mp_raw = dl.load_steel_mapping()
print("shape:", mp_raw.shape)
display(pp.profile(mp_raw[pp.MAPPING_STATIC]))
print("Date / price-history columns (2016-12 → 2018-04), not exported:", [str(c) for c in mp_raw.columns if c not in pp.MAPPING_STATIC][:8], "…")

In [ ]:
mp = pp.mapping_clean()
print("Duplicate 'Unique ID' (reference × plant):", mp.unique_id.duplicated().sum(),
      "| fully duplicated rows:", mp.duplicated().sum())
display(mp[mp.unique_id.duplicated(keep=False)].sort_values("unique_id")[["unique_id", "plant_name", "index", "weight", "grade_in_rfq"]].head(10))
display(mp.groupby(["index", "panel_series"]).size().sort_values(ascending=False).rename("n_refs").reset_index())
print("Lead time distribution:", mp.lead_time_months.value_counts(dropna=False).to_dict())
print("Plants:", mp.plant_name.nunique())

- **(e)** The four main indices (**HDG Benelux, CR France, CR Spain, HDG Germany**) cover most of the references. **These are the series to forecast if we want to speak the buyers' language.** We add `panel_series` to link each index to its panel column (Benelux → MEPS Belgium, since CRU has no Benelux quote).
- **Lead time is 1 month wherever it is filled.** A forecast at *t* therefore drives the price paid at *t+1*.
- **(d)** A few `Unique ID`s (part × plant) appear twice. Several references are blank in many columns (`Standard Batch Quantity` is fully empty).

### 6.7 Snapshot sheets: `Q4-2017`, `Q2-2018`, `UPDATE`, `Weight Calculation`
These are **historical negotiation artefacts**:
- `Q4-2017` and `Q2-2018` give the negotiated monthly or quarterly price changes by index and grade;
- `UPDATE` is a 2016–17 allocation and update tracker by plant and supplier. It contains **employee names**, which are not displayed here;
- `Weight Calculation` is an empty calculator template (length × width × thickness × density).

In [ ]:
for s in ["Q4-2017", "Q2-2018"]:
    t = pd.read_excel(dl.STEEL_XLSX, s, header=2).dropna(how="all", axis=1).dropna(how="all")
    print(s, t.shape); display(t.head(6))
up = pd.read_excel(dl.STEEL_XLSX, "UPDATE", header=2)
print("UPDATE: shape", up.shape, "| references by first allocation:",
      up.groupby(up.columns[1])[up.columns[2]].sum(numeric_only=True).to_dict())
print("Weight Calculation:", pd.read_excel(dl.STEEL_XLSX, "Weight Calculation").to_dict("records"))

**(f) Decision for the whole workbook.**
- **Panel:** CRU Flat domestic HR/CR/HDG (DE, FR, IT, ES, UK), MEPS HR/CR/HDG/EG (DE, FR, IT, ES, UK, BE), CRU Stainless total, base and alloy (5 countries × 3 grades), and CRU Long.
- **Table:** `steel_index_mapping.parquet`, with the static columns plus `panel_series`.
- **Dropped:**
  - the weekly quotes (we keep the average);
  - EG CRU and the Italy imports;
  - the Nordic/Sweden fragments;
  - the ▲ sheets (redundant);
  - the snapshot sheets (no modelling value);
  - the placeholder rows up to 2028.

## 7 · `LME_Copper_Aluminium_EU_Monthly_2016_2026.xlsx`

### (a) Business
This workbook gives London Metal Exchange **copper and aluminium**:
- the **cash** price (spot) and the **3-month** price (forward), in USD and EUR;
- the **cash–3M spread**:
  - **contango** (3M > cash) means the market is well supplied;
  - **backwardation** (cash > 3M) means it is tight now;
- the **LME warehouse stocks**.

The spread and the stocks are classic predictors of metal returns, and the 3M price is the market's own 3-month forecast. The workbook also adds **Nov-2026 month-to-date**, one month beyond the core set.

In [ ]:
xl = pd.ExcelFile(dl.LME_XLSX); print("sheets:", xl.sheet_names)
lme = dl.load_lme()
display(pp.profile(lme.reset_index()))
print(lme["Period status"].value_counts().to_string()); print(lme["FX coverage"].value_counts().to_string())
print("\nMethod & Sources sheet:"); print(pd.read_excel(xl, "Method & Sources", header=None).fillna("").to_string(max_colwidth=110))

In [ ]:
print("duplicate months:", lme.index.duplicated().sum())
comp = lme[lme["Period status"].eq("Completed month")]
fxd = (comp["USD per EUR"] / rm.S06.reindex(comp.index) - 1).abs()
print(f"'USD per EUR' vs S06 (ECB): median rel. diff {fxd.median():.1e}, max {fxd.max():.1e}")
g = pd.concat({"Cu": comp["Cu cash EUR/t"] / rm.D01 - 1, "Al": comp["Al cash EUR/t"] / rm.D02 - 1}, axis=1).dropna() * 100
print("LME workbook vs challenge D01/D02 (%):"); display(g.describe().loc[["mean", "std", "min", "max"]].round(3))
print("EUR columns == USD / FX:", np.allclose(comp["Cu cash EUR/t"], comp["Cu cash USD/t"] / comp["USD per EUR"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
for ax, m in zip(axes, ["Cu", "Al"]):
    rel = 100 * (lme[f"{m} 3m USD/t"] / lme[f"{m} cash USD/t"] - 1)
    ax.bar(rel.index, rel.values, width=25, color=np.where(rel > 0, PAL[0], PAL[7]))
    ax.set_title(f"{m}: 3M vs cash (%): >0 contango, <0 backwardation")
plt.show()
lme.tail(3)

- **(d)** `USD per EUR` is S06 (Westmetall's copy of the ECB fixings, differing by at most about 0.06 %), and the EUR columns are USD/FX. Both are redundant. The cash prices match D01/D02 within about 0.1 %, so **the sources are consistent**.
- **(e)** **Copper is currently in backwardation** (cash > 3M). The market is tight, and this agrees with the all-time-high price. The **Nov-2026 row is month-to-date and provisional**.
- **(f)** The panel keeps the completed months and adds `LME_*` cash/3M (USD, EUR), spreads, stocks and the relative spreads. FX is dropped (duplicate). The Nov MTD row goes to `beyond_panel_end.parquet`. The LME cash price is used to **nowcast S01/S02 for Oct-2026**.

## 8 · `Epoxy for student.xlsx`

### (a) Business
**Epoxy resin** is the insulating material in RMUs: cast-resin bushings, insulators, support barriers. The kick-off deck asks for it under "insulating materials". The file is a **cost build-up**. The chemistry is benzene + propylene → phenol + acetone → bisphenol-A, then BPA + **epichlorohydrin (ECH)** → epoxy, plus other costs. So the resin price should follow its feedstocks.

In [ ]:
ep_raw = pd.read_excel(dl.EPOXY_XLSX)
print("raw columns:", ep_raw.columns.tolist())
ep = dl.load_epoxy()
display(pp.profile(ep.reset_index()))
display(pp.series_coverage(ep))
print("duplicate months:", ep.index.duplicated().sum())
ep.loc["2026-03":"2026-07"]

In [ ]:
r = np.log(ep).diff()
z = (r - r.mean()) / r.std()
print("Largest |z| monthly moves:"); display(z.abs().stack().sort_values(ascending=False).head(6))
clean = pp.epoxy_block()[0]
fit = clean.dropna()
X = np.c_[fit[["EPOXY_BENZENE", "EPOXY_ACETONE", "EPOXY_ECH", "EPOXY_OTHER"]].values, np.ones(len(fit))]
coef, *_ = np.linalg.lstsq(X, fit.EPOXY_RESIN.values, rcond=None)
pred = X @ coef
print("Recipe fit resin ≈ a·benzene + b·acetone + c·ECH + d·other + k:", dict(zip(["benzene", "acetone", "ECH", "other", "const"], coef.round(3))),
      f"R² = {1 - ((fit.EPOXY_RESIN - pred) ** 2).sum() / ((fit.EPOXY_RESIN - fit.EPOXY_RESIN.mean()) ** 2).sum():.3f}")
fig, ax = plt.subplots(figsize=(11, 3.4))
for i, c in enumerate(ep.columns):
    ax.plot(ep.index, ep[c], color=PAL[i], label=c)
ax.annotate("May-2026: ECH = 257 (typo)", xy=(pd.Timestamp("2026-05-01"), ep.loc["2026-05-01", "Epychlorhydrine"]),
            xytext=(pd.Timestamp("2023-01-01"), 600), arrowprops=dict(arrowstyle="->", color=INK2), fontsize=8)
ax.legend(ncol=5); ax.set_ylabel("EUR/t"); ax.set_title("Epoxy resin and feedstocks"); plt.show()

- **(e) Data-entry error.** **May-2026 ECH is about 10× below** the neighbouring months, and the resin price that month is computed from it. Left in, it creates a spurious −90 % / +900 % month. We **set both cells to NaN** in the raw panel and **interpolate them** in the filled panel (`interp_data_error`).
- The resin is an almost **deterministic linear recipe** of its feedstocks (R² ≈ 0.98). So **forecast the feedstocks, then derive the resin**.
- **(c)** It covers Jan-2018 → Oct-2026 with no other holes.

## 9 · Teardown should-cost benchmark (`benchmark-modules-fonctionnels-rm6-…-sf6.xlsx` + `.pdf`)

### (a) Business
This is a teardown **should-cost** of the Schneider **RM6 3-function module** against Ormazabal, SGC, ABB, Siemens and Eaton. It answers the question: *what should each competitor's unit cost to make, given its BOM?* It splits the cost into material, process, labour and overhead. It is **marked provisional**, and most part masses are *inferred*. The PDF is the printed version.

In [ ]:
xl = pd.ExcelFile(dl.BENCH_XLSX); print("sheets:", xl.sheet_names)
raw = pd.read_excel(xl, "Benchmark", header=None)
print("header notes:", raw.iloc[1:3, 0].tolist())
print("All 'Field' rows of the Benchmark sheet:", raw.iloc[5:, 0].dropna().tolist())
bench = pp.competitor_benchmark()
display(bench)
display(pp.profile(bench))

In [ ]:
tb = pp.load_teardown_boms()
display(pp.profile(tb))
print("duplicate (product, part):", tb.duplicated(["product", "part"]).sum(), "| share of parts with inferred mass:", round(tb.mass_inferred.mean(), 2))
mass = tb.pivot_table(index="product", columns="category", values="total_mass_g", aggfunc="sum").div(1000).round(1)
mass["TOTAL_kg"] = mass.sum(axis=1)
display(mass)
print("Price sources sheet:"); display(pd.read_excel(xl, "Price sources", header=1).dropna(how="all").head(12))

- **(e) SGC is broken**: its total is **two orders of magnitude below** every other unit. It is flagged `is_valid = False` and must be excluded.
- **Materials are about 80 % of should-cost** in every valid unit, so a raw-material move passes almost one-for-one into competitive cost. This is the business bridge for the recommendation.
- **Eaton Xiria is the only SF6-free unit** (air/vacuum), and its cost looks low. Treat it with caution.
- **Every part mass is inferred** ("total held to the mass budget"). For Schneider the part-level total (Σ mass × qty) is more than twice the benchmark sheet's `Mass (g)`, and SGC's parts add up to almost nothing. So these are **order-of-magnitude** numbers.
- Only one list price was found, for Siemens, from a single web source. Several price sources were rejected: used listings, unreadable pages.
- **(f)** Exported as `competitor_benchmark.parquet` (one row per product) and `teardown_bom.parquet` (one row per part).

## 10 · `RMU_IQI_EU_Price_Benchmark_2017-2026_Synthetic.xlsx` (unlocked copy, 18-Nov-2026)

### (a) Business
These are **synthetic** ("AI/ML") **yearly selling prices** of the same RMU class in the EU, by brand, with a P10/P50/P90 band in EUR ex-VAT. Comparing them with material cost shows how much of the price is raw material, i.e. how much a commodity shock moves the price.

In [ ]:
xl = pd.ExcelFile(dl.PRICE_XLSX); print("sheets:", xl.sheet_names)
pb = dl.load_price_benchmark()
display(pp.profile(pb))
print("duplicate (brand, year):", pb.duplicated(["brand", "year"]).sum(),
      "| P10 <= P50 <= P90 everywhere:", bool(((pb.p10 <= pb.p50) & (pb.p50 <= pb.p90)).all()))
p50 = pb.pivot(index="year", columns="brand", values="p50")
display((p50.pct_change() * 100).round(1).tail(4))
cd = pd.read_excel(xl, "Chart_Data", header=None)
print("Chart_Data:", cd.iat[0, 0], "| SF6 marker x =", cd.iat[3, 19])

- **(e)** There is a **2026 jump**: SF6 brands are up **+32–40 %**, while Eaton, the only SF6-free unit, is up +8 %. This matches the **EU F-gas Regulation 2024/573** ban on SF6 in new MV switchgear up to 24 kV from 2026. It is a pricing-strategy signal for question 3 (non-regulated markets).
- **(d)** `Chart_Data` only links back to `ML_Trend` (chart helper), so it is redundant.
- **(f)** Exported as `price_benchmark.parquet`. It is yearly and synthetic, so use it for business framing, not as a model target.

## 11 · `Supplier_balancesheet.zip`: PitchBook financials for 9 suppliers

### (a) Business
These are financial statements (Income Statement, Balance Sheet, Ratios, sometimes Cash Flow) in kEUR for 9 suppliers. They support a **supplier-risk overlay**: which suppliers can absorb a raw-material price shock, and which might fail or force a price increase?

In [ ]:
for f in sorted(dl.SUPPLIERS.glob("*.xlsx")):
    print(f"{dl._supplier_name(f):35s}", pd.ExcelFile(f).sheet_names)
fin = dl.load_supplier_financials()
display(pp.profile(fin))
print("Coverage (first–last year, n datapoints) per supplier × statement:")
fin.groupby(["supplier", "statement"]).year.agg(lambda y: f"{y.min()}–{y.max()} ({len(y)})").unstack()

In [ ]:
print("exact duplicate rows:", fin.duplicated().sum(),
      "| duplicate (supplier, statement, item, period):", fin.duplicated(["supplier", "statement", "item", "period"]).sum())
print("Revenue label variants:", fin[fin.item.isin(["Revenue", "Total Revenue"]) & fin.statement.eq("Income Statement")].groupby("item").supplier.unique().map(list).to_dict())
print("PitchBook 'Debt to Equity' distinct values:", sorted(fin.loc[fin.item == "Debt to Equity", "value"].unique())[:5])
fin_c, kpis = pp.supplier_tables()
kpis

- **(e) Trap:** PitchBook's own **"Debt to Equity" is 0 for 8 of the 9 suppliers**, including the largest supplier, which has material debt. We recompute it as Total Debt / Total Equity. A debt of 0 can mean "not reported".
- Revenue is labelled `Revenue` or `Total Revenue` depending on the template, so we harmonise the two labels.
- Supplier names and figures are not shown (NDA). Flags: the **largest supplier is highly leveraged** (debt far above EBITDA); **one has negative EBITDA**; **one has a stale P&L** (several years old); **one has a balance sheet only**.
- **(f)** Exported as `supplier_financials.parquet` (long) and `supplier_kpis.parquet` (latest snapshot plus flags). Coverage is too uneven for statistics; use them as a qualitative overlay.

## 12 · PDFs (not parsed)

| File | Business content |
|---|---|
| `X_HEC_2026_Pres.pdf` (kick-off deck) | ETO contracts lock prices for 3, 6 or 9 months. AirSeT has a higher cost and a thinner margin. Forecast **all** RM inputs (copper, carbon steel, insulating materials = epoxy, electronics: no series given) and carry them into **2027 RMU pricing**. Question 3 covers non-regulated markets (APAC, Middle East, Americas). Accuracy and strategy count equally in the evaluation. The deck contains presenters' personal data, so keep it out of the repo. |
| `benchmark-modules-…-sf6.pdf` | Printed version of the teardown workbook (§9). |

## 13 · Preprocessing & export to parquet

The whole pipeline lives in [`src/preprocessing.py`](../src/preprocessing.py) → `pp.run_pipeline()`:

1. **Blocks** go onto one **month-start grid, 2012-01 → 2026-10** (178 rows): RM proxies, energy, CRU Flat, MEPS, CRU Stainless, CRU Long, LME and epoxy. The core S/E series start in Nov-2016; CRU/MEPS go back to 2012. Before its first observation a series is NaN, never back-filled.
2. **Cleaning:**
   - drop L01–L08 (empty), E23 (= S06), the LME FX column (= S06), EG CRU, the Italy imports and the Nordic/Sweden MEPS fragments;
   - set the epoxy May-2026 typo to NaN;
   - keep the LME Nov MTD row and the CRU Nov quotes in `beyond_panel_end` only.
3. **Imputation** (filled panel only, every cell flagged):
   1. `interp_internal` / `interp_data_error`: linear interpolation of single-month holes;
   2. `lme_proxy`: S01/S02 for Oct-2026, from LME cash × the 12-month median ratio;
   3. `identity`: D01, D02 and ED* recomputed from their components;
   4. `ffill`: the remaining ragged edge, **only if the whole trailing gap is ≤ 3 months**.

   Lagging or discontinued series stay NaN.
4. **Catalog:**
   - source, unit and family;
   - `role` (target / driver, matching `src/features.py`);
   - `pub_lag_months`, the publication lag to use when lagging features to avoid look-ahead;
   - `bom_material`;
   - coverage, `status` and `n_imputed`.

In [ ]:
T = pp.run_pipeline()
raw, filled, method, cat = T["panel_monthly"], T["panel_monthly_filled"], T["panel_impute_method"], T["series_catalog"]
print("panel:", raw.shape, raw.index.min().date(), "→", raw.index.max().date())
display(cat.groupby(["family", "status"]).size().unstack(fill_value=0))
display(method.stack().replace("", "observed / NaN").value_counts().rename("cells"))

In [ ]:
# Which cells were imputed, and how?
imp = method.stack(); imp = imp[imp != ""]
imp.rename("method").reset_index().rename(columns={"level_1": "series"}).groupby(["method", "month"]).series.apply(
    lambda s: ", ".join(s) if len(s) <= 8 else f"{len(s)} series").to_frame()

In [ ]:
plot_missing(filled.loc["2016-01":], "Filled panel (2016 → 2026-10) · the series still missing at the end are lagging / discontinued", max_cols=172)

### Sanity checks (asserts)

In [ ]:
idx = raw.index
assert idx.equals(pd.date_range(pp.PANEL_START, pp.PANEL_END, freq="MS")), "grid has gaps"
assert not raw.columns.duplicated().any()
# raw panel = loaders, untouched (spot checks)
assert np.allclose(raw.D01.dropna(), dl.load_rm_wide().D01.dropna())
assert np.allclose(raw.CRU_CR_DE.dropna(), fe._cru(dl.load_cru_flat(), country="Germany", product="CR", market="Domestic").reindex(raw.CRU_CR_DE.dropna().index))
# filled never overwrites an observed value and never fills before the first observation
obs = raw.notna()
assert np.allclose(filled[obs].fillna(0), raw[obs].fillna(0))
for c in raw:
    fv = raw[c].first_valid_index()
    assert filled[c].loc[:fv].iloc[:-1].isna().all(), c
assert (method.ne("") == (filled.notna() & raw.isna())).all().all(), "method flags inconsistent"
# identities hold in the filled panel
assert np.allclose((filled.D01 / (filled.S01 / filled.S06)).dropna(), 1, rtol=0, atol=1e-4)  # S06 is rounded to 4 dp in the CSV
# long table keys unique
lg = T["panel_monthly_long"]
assert not lg.duplicated(["month", "series"]).any()
# every mapping index has a panel series
assert set(T["steel_index_mapping"].panel_series.dropna()) <= set(raw.columns)
# compatibility with src/features.build_panel()
tg, ex = fe.build_panel()
for c in tg.columns:
    pc = pp.FEATURES_RENAME.get(c, c)
    a, b = tg[c].dropna(), raw[pc].reindex(tg[c].dropna().index)
    assert np.allclose(a, b), c
print("✓ all checks passed: grid complete, raw == loaders, imputation only on NaN cells, identities hold,",
      "long keys unique, mapping → panel complete, features.build_panel() targets reproduced")

### Write `data/processed/*.parquet`

In [ ]:
written = pp.write_processed(T)
written

In [ ]:
# Reload and show schemas
for f in written.file:
    df = pd.read_parquet(ROOT / f)
    print(f"{f:50s} {str(df.shape):>12s}  {', '.join(map(str, df.columns[:8]))}{' …' if df.shape[1] > 8 else ''}")
pd.read_parquet(ROOT / "data/processed/series_catalog.parquet").query("role == 'target'")[
    ["family", "unit", "first", "last", "pub_lag_months", "bom_material", "status", "n_imputed"]]

## 14 · How to use the outputs

```python
import pandas as pd
P = "data/processed/"
raw    = pd.read_parquet(P + "panel_monthly.parquet")          # wide, NaN kept: use for backtests (no imputed values)
filled = pd.read_parquet(P + "panel_monthly_filled.parquet")   # wide, ragged edge nowcast: use for the production forecast
how    = pd.read_parquet(P + "panel_impute_method.parquet")    # "" = observed, else the imputation method
cat    = pd.read_parquet(P + "series_catalog.parquet")         # role, family, unit, pub_lag_months, bom_material, status
long   = pd.read_parquet(P + "panel_monthly_long.parquet")     # tidy, for global / panel models
targets = cat.index[cat.role == "target"]                       # same 12 targets as src/features.py
```

**Modelling rules this data implies:**
1. **Model Δlog, not levels.** Everything is I(1), and indices have different bases.
2. **Shift drivers by `pub_lag_months`** to avoid look-ahead. In real time, the Sep PPI is not known in early October.
3. **Backtest on `panel_monthly`, not on the filled panel.** Imputed cells are *estimates* and must never be scored as truth.
4. Do not feed exact duplicates as separate signals:
   - D01 ↔ S01/S06;
   - ED* ↔ E*;
   - MEPS ↔ CRU;
   - LME cash ↔ S01/S02.
5. Treat UK series as GBP. Use `status = discontinued` series only as history.